# Libraries

In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact, FloatSlider, Button, VBox, HBox, Output
from IPython.display import display, clear_output
from skimage.restoration import denoise_tv_chambolle
from scipy.signal import convolve2d
import sys
import os
import niqe 

# TWIST Optimisation, Masking, and PSF

In [2]:
def generate_psf(size):
    psf = np.ones((size, size), dtype=np.float32)
    return psf / psf.sum()

def generate_mask(shape):
    mask = np.ones(shape, dtype=np.float32)
    for i in range(0, shape[0], 5):
        for j in range(0, shape[1], 5):
            mask[i, j] = 0.5 + 0.5 * np.random.rand()
    return mask

def generate_airy_psf(size, cutoff_freq=0.3):
    center = size // 2
    y, x = np.ogrid[:size, :size]
    fx = (x - center) / size
    fy = (y - center) / size
    f = np.sqrt(fx**2 + fy**2)
    f = np.where(f == 0, 1e-10, f)
    fc = cutoff_freq
    ratio = np.clip(f / fc, 0, 1)
    otf = np.zeros_like(f)
    valid_mask = ratio <= 1.0
    arccos_term = np.arccos(ratio[valid_mask])
    otf[valid_mask] = (1/np.pi) * (2 * arccos_term - np.sin(2 * arccos_term))
    otf_shifted = np.fft.fftshift(otf)
    psf = np.real(np.fft.ifft2(otf_shifted))
    psf = np.fft.fftshift(psf)
    psf = np.abs(psf)
    psf = psf / psf.sum()
    return psf.astype(np.float32)

def estimate_transmittance_matrix(uniform_image, psf, delta=0.01):
    psf_padded = np.zeros_like(uniform_image)
    h_offset = (uniform_image.shape[0] - psf.shape[0]) // 2
    w_offset = (uniform_image.shape[1] - psf.shape[1]) // 2
    psf_padded[h_offset:h_offset+psf.shape[0], w_offset:w_offset+psf.shape[1]] = psf
    F_h = np.fft.fft2(psf_padded)
    F_yM = np.fft.fft2(uniform_image)
    H_conj = np.conj(F_h)
    H_mag_sq = np.abs(F_h)**2
    denominator = np.where(H_mag_sq + delta == 0, delta, H_mag_sq + delta)
    M_freq = (H_conj * F_yM) / denominator
    M = np.real(np.fft.ifft2(M_freq))
    M = np.abs(M)
    return np.clip(M, 0.1, 2.0).astype(np.float32)

def create_uniform_reference_image(shape, noise_level=0.05):
    uniform = np.ones(shape, dtype=np.float32) * 0.8
    for i in range(0, shape[0], 8):
        for j in range(0, shape[1], 8):
            uniform[i:i+6, j:j+6] *= (0.7 + 0.4 * np.random.rand())
    noise = np.random.normal(0, noise_level, shape)
    return np.clip(uniform + noise, 0, 1)

def compute_tv_norm(x):
    dx = np.diff(x, axis=1, append=x[:, -1:])
    dy = np.diff(x, axis=0, append=x[-1:, :])
    return np.sum(np.sqrt(dx**2 + dy**2))

def A(x, h, m):
    return convolve2d(m * x, h, mode='same', boundary='symm')

def AT(r, h, m):
    return m * convolve2d(r, h[::-1, ::-1], mode='same', boundary='symm')

def residual(x, y, h, m, lamb):
    fidelity = 0.5 * np.linalg.norm(y - A(x, h, m))**2
    return fidelity + lamb * compute_tv_norm(x)

def twist_restore(gray, psf_airy, mask_estimated, lambda_tv, alpha, beta, max_iter=30, tol=1e-4):
    y = A(gray, psf_airy, mask_estimated)
    x0 = np.copy(y)
    r0 = y - A(x0, psf_airy, mask_estimated)
    x1 = denoise_tv_chambolle(x0 + AT(r0, psf_airy, mask_estimated), weight=lambda_tv, channel_axis=None)
    R_prev = residual(x1, y, psf_airy, mask_estimated, lambda_tv)
    for _ in range(max_iter):
        r = y - A(x1, psf_airy, mask_estimated)
        v = x1 + AT(r, psf_airy, mask_estimated)
        v_denoised = denoise_tv_chambolle(v, weight=lambda_tv, channel_axis=None)
        x_new = (1 - alpha) * x0 + (alpha - beta) * x1 + beta * v_denoised
        R_new = residual(x_new, y, psf_airy, mask_estimated, lambda_tv)
        if R_new > R_prev:
            r_fallback = y - A(x1, psf_airy, mask_estimated)
            x_new = denoise_tv_chambolle(x1 + AT(r_fallback, psf_airy, mask_estimated), weight=lambda_tv, channel_axis=None)
            R_new = residual(x_new, y, psf_airy, mask_estimated, lambda_tv)
        if abs(R_new - R_prev) / R_prev < tol:
            break
        x0, x1 = x1, x_new
        R_prev = R_new
    return np.clip(x_new, 0, 1)

# Directory Setup

In [3]:
INPUT_FOLDER = r"C:\Users\makvanmt1\Downloads\FInal Data set\Gridded images"
OUTPUT_FOLDER = os.path.join(INPUT_FOLDER, "TWIST_Restored_Results_final")

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

all_files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif'))]

if all_files:
    # Load first image for the GUI preview
    preview_frame = cv2.imread(os.path.join(INPUT_FOLDER, all_files[0]))
    gray = cv2.cvtColor(preview_frame, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    psf_airy = generate_airy_psf(7, cutoff_freq=0.3)
    uniform_ref = create_uniform_reference_image(gray.shape)
    mask_estimated = estimate_transmittance_matrix(uniform_ref, psf_airy)
    print(f"Ready to process {len(all_files)} images.")

Ready to process 22 images.


# Interactive GUI Setup, NIQE Evaluation, and Batch Processing

In [ ]:
out = Output()

def on_niqe_click(b):
    with out:
        clear_output()
       
        restored = twist_restore(gray, psf_airy, mask_estimated, l_slide.value, a_slide.value, beta=0.5)
        
        h, w = gray.shape
        if h < 192 or w < 192:
            niqe_in = cv2.resize(gray, (max(w, 192), max(h, 192)), interpolation=cv2.INTER_CUBIC)
            niqe_out = cv2.resize(restored, (max(w, 192), max(h, 192)), interpolation=cv2.INTER_CUBIC)
        else:
            niqe_in, niqe_out = gray, restored
            
        score_in = niqe.niqe((niqe_in * 255).astype(np.float64))
        score_out = niqe.niqe((niqe_out * 255).astype(np.float64))
        print(f"NIQE Preview (Lower is Better)\nInput: {score_in:.2f}\nOutput: {score_out:.2f}")

def on_batch_click(b):
    with out:
        clear_output()
        records = []
        lamb, alpha = l_slide.value, a_slide.value
        csv_path = os.path.join(OUTPUT_FOLDER, "NIQE_Batch_Report.csv")
        
        print(f"Starting Batch: {len(all_files)} images. Saving CSV every 50 images...")

        for i, filename in enumerate(all_files):
            try:
                img = cv2.imread(os.path.join(INPUT_FOLDER, filename))
                g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
                
                m_curr = estimate_transmittance_matrix(create_uniform_reference_image(g.shape), psf_airy)
                
                res = twist_restore(g, psf_airy, m_curr, lamb, alpha, beta=0.5)
                
                cv2.imwrite(os.path.join(OUTPUT_FOLDER, f"restored_{filename}"), (res * 255).astype(np.uint8))
                
                # NIQE Logic
                h, w = g.shape
                n_in = cv2.resize(g, (max(w, 192), max(h, 192)), interpolation=cv2.INTER_CUBIC) if (h<192 or w<192) else g
                n_out = cv2.resize(res, (max(w, 192), max(h, 192)), interpolation=cv2.INTER_CUBIC) if (h<192 or w<192) else res
                
                score_orig = niqe.niqe((n_in * 255).astype(np.float64))
                score_rest = niqe.niqe((n_out * 255).astype(np.float64))
                
                records.append({
                    "Filename": filename,
                    "Original_NIQE": round(float(score_orig), 2),
                    "Restored_NIQE": round(float(score_rest), 2)
                })

                # Save every 50 images
                if len(records) % 50 == 0:
                    pd.DataFrame(records).to_csv(csv_path, index=False)
                    print(f"Progress: {len(records)}/{len(all_files)} - CSV Updated.")

            except Exception as e:
                print(f"Error on {filename}: {e}")

        # Final save for the remaining images
        pd.DataFrame(records).to_csv(csv_path, index=False)
        print(f"\nSuccess! All processed results saved to: {csv_path}")

def preview(lambda_tv, alpha):
    # Explicitly passing beta=BETA
    restored = twist_restore(gray, psf_airy, mask_estimated, lambda_tv, alpha, beta= 0.5)
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    axs[0].imshow(gray, cmap='gray'); axs[0].set_title("Input Preview"); axs[0].axis('off')
    axs[1].imshow(restored, cmap='gray'); axs[1].set_title("TWIST Preview"); axs[1].axis('off')
    plt.tight_layout(); plt.show()

l_slide = FloatSlider(value=0.14, min=0.01, max=1.0, step=0.01, description='λ (Smooth)')
a_slide = FloatSlider(value=1.7, min=0.0, max=2.0, step=0.1, description='α')

niqe_btn = Button(description="Calculate NIQE", button_style='info')
batch_btn = Button(description="Process All 3500 Images", button_style='danger')

niqe_btn.on_click(on_niqe_click)
batch_btn.on_click(on_batch_click)

interact(preview, lambda_tv=l_slide, alpha=a_slide)
display(VBox([HBox([niqe_btn, batch_btn]), out]))

interactive(children=(FloatSlider(value=0.06, description='λ (Smooth)', max=1.0, min=0.01, step=0.01), FloatSl…